# 🚀 ATML PA3: Large Language Models, Vision-Language Grounding & Test-Time Compute
## Master Kaggle Runner (GPU Orchestrator)

This notebook is the end-to-end execution harness designed to run all experiments for **ATML PA3** (EE-5102 / CS-6304, Fall 2026 LUMS) on a Kaggle GPU instance (e.g. Dual T4 or P100).

### Instructions for Running on Kaggle:
1. **Accelerator:** Under Notebook Settings, select **GPU T4 x2** or **GPU P100**.
2. **Internet:** Ensure Internet access is toggled **ON** (to download HuggingFace models & datasets).
3. **Persistence:** Output checkpoints and figures will be saved in `/kaggle/working/`.


In [ ]:
# 1. Environment & Hardware Diagnostics
import os, sys, time
import torch

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)} | Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("WARNING: CUDA is not available. Please switch notebook accelerator to GPU in settings.")


In [ ]:
# 2. Install Required Packages
!pip install -q transformers datasets accelerate peft evaluate sentencepiece seaborn scikit-learn
print("Dependencies installed successfully.")


In [ ]:
# 3. Setup Working Directory & Repository Layout
import os, sys
from pathlib import Path

# Detect Kaggle environment
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')
ROOT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('.').resolve()
os.chdir(ROOT_DIR)
sys.path.insert(0, str(ROOT_DIR))

# Create standard directories
os.makedirs("task1_llm_finetuning/checkpoints", exist_ok=True)
os.makedirs("task1_llm_finetuning/results/figures", exist_ok=True)
os.makedirs("task1_llm_finetuning/splits", exist_ok=True)

print(f"Working Directory: {os.getcwd()}")


---
## Task 1 — Step 1: Data Curation & Activation Probe Manifest (Seed 6304)
Filter 40k MetaMathQA training samples with strict three-tier exclusion:
1. Empty query or response.
2. Formatted token length > 2,048 tokens.
3. Overlap with GSM8K test set (1,319 questions) after text normalization.
Sample 32 GSM8K test questions and 32 HellaSwag validation contexts for activation probing.


In [ ]:
# Run Task 1 Data Preparation & Probe Sampling
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from shared.src.seed import set_seed, SEED
set_seed(6304)

from task1_llm_finetuning.src.data import prepare_metamath_dataset, sample_activation_probe_set
from datasets import load_dataset
from transformers import AutoTokenizer

print("Loading Qwen/Qwen3-0.6B tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Preparing MetaMathQA 40,000 dataset...")
dataset, stats = prepare_metamath_dataset(
    tokenizer=tokenizer,
    target_count=40000,
    max_length=2048,
    seed=6304,
)
print("Filtering statistics:", stats)

print("Sampling activation probe set (Seed 6304)...")
try:
    gsm_test = load_dataset("gsm8k", "main", split="test")
    hella_val = load_dataset("hellaswag", split="validation")
    sample_activation_probe_set(
        gsm8k_questions=[{"id": i, "question": ex["question"]} for i, ex in enumerate(gsm_test)],
        hellaswag_contexts=[{"id": i, "ctx": ex["ctx"]} for i, ex in enumerate(hella_val)],
        seed=6304,
        n_samples=32,
        save_path="task1_llm_finetuning/splits/probe_set_seed6304.json"
    )
    print("Probe set saved to task1_llm_finetuning/splits/probe_set_seed6304.json")
except Exception as e:
    print("Probe sampling warning (will use pre-existing manifest if offline):", e)


---
## Task 1 — Step 2: Execute Fine-Tuning Runs (1 Epoch = 1,250 Updates)
Train the 5 fine-tuning variants specified in the manual:
1. **Full FT** (`task1_llm_finetuning/configs/full_ft.yaml`)
2. **Full FT + L2-SP** ($\lambda=10^{-3}$, `task1_llm_finetuning/configs/full_ft_l2sp.yaml`)
3. **LoRA-4** ($r=4, lpha=8$, `task1_llm_finetuning/configs/lora_r4.yaml`)
4. **LoRA-16** ($r=16, lpha=32$, `task1_llm_finetuning/configs/lora_r16.yaml`)
5. **OPLoRA-16** ($r=16, lpha=32, k=128$, `task1_llm_finetuning/configs/oplora_r16.yaml`)

*Toggle `DEBUG_MODE = True` to run 10 updates for rapid smoke-testing before launching full runs.*


In [ ]:
# Training Orchestrator
DEBUG_MODE = False  # Set to True for quick 10-step sanity check; False for full 1,250 updates
RUN_TRAINING = True # Set to False if checkpoints already exist

configs_to_run = [
    "task1_llm_finetuning/configs/lora_r4.yaml",
    "task1_llm_finetuning/configs/lora_r16.yaml",
    "task1_llm_finetuning/configs/oplora_r16.yaml",
    "task1_llm_finetuning/configs/full_ft.yaml",
    "task1_llm_finetuning/configs/full_ft_l2sp.yaml",
]

if RUN_TRAINING and torch.cuda.is_available():
    for cfg_path in configs_to_run:
        print(f"\n{'='*30}\nStarting training run: {cfg_path}\n{'='*30}")
        debug_flag = "--debug" if DEBUG_MODE else ""
        !python -m task1_llm_finetuning.src.train --config {cfg_path} {debug_flag} --save_dir task1_llm_finetuning/checkpoints
else:
    print("Skipping live training. Will use saved checkpoints or offline logs.")


---
## Task 1 — Step 3: Zero-Shot Benchmark Evaluations
Evaluate the starting model and all fine-tuned checkpoints across:
1. **GSM8K** (greedy generation, 1,024 token limit, flexible EM)
2. **HellaSwag** (candidate likelihood, acc_norm)
3. **BoolQ** (candidate likelihood, acc)
4. **ARC-Easy** (candidate likelihood, acc_norm)


In [ ]:
# Benchmark Evaluation Harness
RUN_EVALS = False # Set to True to run generation on all 1,319 GSM8K test questions
EVAL_LIMIT = 50 if DEBUG_MODE else None

if RUN_EVALS and torch.cuda.is_available():
    print("Evaluating baseline starting model...")
    !python -m task1_llm_finetuning.src.evaluate --model_name Qwen/Qwen3-0.6B --output_csv task1_llm_finetuning/results/eval_baseline.csv
    
    # Evaluate fine-tuned checkpoints
    for method in ["lora_r4", "lora_r16", "oplora_r16", "full_ft", "full_ft_l2sp"]:
        ckpt_path = f"task1_llm_finetuning/checkpoints/{method}_update_1250.pt"
        if os.path.exists(ckpt_path):
            print(f"Evaluating {method}...")
            !python -m task1_llm_finetuning.src.evaluate --checkpoint {ckpt_path} --output_csv task1_llm_finetuning/results/eval_{method}.csv
else:
    print("Evaluation harness ready. To run live evaluations, toggle RUN_EVALS = True.")


---
## Task 1 — Step 4: Render All 12 Figures & Master Results Tables
Executes the plotting pipeline, saving 300 DPI publication-grade figures to `task1_llm_finetuning/results/figures/` and master CSV tables to `task1_llm_finetuning/results/`.


In [ ]:
# Run Task 1 Master Figure & Table Generator
!python task1_llm_finetuning/run_task1_figures.py


---
## Task 1 — Step 5: Visual Inspection of Results
Display key figures directly in the notebook:


In [ ]:
from IPython.display import Image, display

figures_to_show = [
    "task1_1_decontamination_and_lengths.png",
    "task1_2_sft_convergence_curves.png",
    "task1_3_pareto_frontier.png",
    "task1_4_subspace_interference_and_magnitude.png",
    "task1_5_format_compliance_and_errors.png",
]

for fig_name in figures_to_show:
    path = f"task1_llm_finetuning/results/figures/{fig_name}"
    if os.path.exists(path):
        print(f"Displaying {fig_name}:")
        display(Image(filename=path, width=850))


---
## Archive & Download Outputs
Bundle all results, figures, and lightweight adapter checkpoints for submission or local inspection:


In [ ]:
!zip -r atml_pa3_task1_artifacts.zip task1_llm_finetuning/results task1_llm_finetuning/splits
print("Saved archive to atml_pa3_task1_artifacts.zip! You can download this file from Kaggle output.")
